# Streaming Operator Inference for the Viscous Burgers' Equation

This notebook demonstrates **Streaming Operator Inference (Streaming-OpInf)**, which learns the same reduced-order models as standard Operator Inference (OpInf) while processing the training data as a *stream*, so that the full snapshot matrix never has to be stored or revisited.

We compare three ways of learning a linear-quadratic reduced model

$$
\dot{\hat{\mathbf{x}}}(t) = \hat{\mathbf{A}}\hat{\mathbf{x}}(t) + \hat{\mathbf{A}}_2\,(\hat{\mathbf{x}}(t) \oslash \hat{\mathbf{x}}(t)) + \hat{\mathbf{B}}u(t),
$$

where $\oslash$ denotes the unique (non-redundant) Kronecker product:

1. **Batch OpInf** (`opinf`): POD basis from the SVD of all snapshots, then one least-squares solve.
2. **Two-pass Streaming-OpInf** (`TwoPassStreamingOpInf`): given a basis, the reduced snapshots are streamed one at a time into a recursive least-squares (RLS) solver that updates the operators on the fly.
3. **One-pass Streaming-OpInf** (`OnePassStreamingOpInf`): every full-order snapshot is seen exactly once; it updates an incremental SVD from which both the POD basis and the reduced data are recovered at the end.

See the [Streaming OpInf manual](https://smallpondtom.github.io/LiftAndLearn.jl/stable/manual/streaming/Overview/) for the theory.

## Setup

In [ ]:
using CairoMakie
using LinearAlgebra
using SparseArrays
using LiftAndLearn
const LnL = LiftAndLearn
const ⊘ = LnL.UniqueKronecker.:⊘   # unique Kronecker product x ⊘ x

## Full-order model

We discretize the periodic viscous Burgers' equation with distributed forcing

$$
v_t = \mu v_{xx} - \left(\tfrac{1}{2}v^2\right)_x + b(x)\,u(t), \qquad x \in [0, 1),
$$

with central finite differences on $n = 128$ grid points. The semi-discrete system has exactly the linear-quadratic structure above with $\mathbf{A}$ (diffusion), a quadratic convection term, and $\mathbf{B} = b(x)$. We integrate it with a semi-implicit Euler scheme (implicit diffusion, explicit convection and forcing) and generate a training trajectory and a test trajectory driven by a different input.

In [ ]:
n  = 128                      # number of grid points (full-order dimension)
Δx = 1 / n
x  = (0:n-1) .* Δx
μ  = 0.02                     # viscosity

# Linear (diffusion) operator: periodic second-order central difference
A = spdiagm(0 => fill(-2.0, n), 1 => ones(n-1), -1 => ones(n-1))
A[1, n] = 1.0; A[n, 1] = 1.0
A = μ / Δx^2 * A

# Input operator: smooth forcing profile
B = reshape(exp.(-((x .- 0.3) ./ 0.08).^2), n, 1)

# Quadratic (convection) term in conservative form: -(v_{i+1}^2 - v_{i-1}^2) / (4Δx)
function convection(v)
    vp = circshift(v, -1)
    vm = circshift(v, 1)
    return -(vp .^ 2 .- vm .^ 2) ./ (4Δx)
end

# Semi-implicit Euler: implicit diffusion, explicit convection and forcing
function integrate_fom(x0, u, Δt)
    K = length(u)
    X = zeros(n, K)
    X[:, 1] = x0
    F = lu(I - Δt * A)
    for k in 1:K-1
        X[:, k+1] = F \ (X[:, k] + Δt * (convection(X[:, k]) + B[:, 1] * u[k+1]))
    end
    return X
end

In [ ]:
Δt = 1e-3
T  = 1.0
t  = 0:Δt:T
K  = length(t)
x0 = 0.5 .* sin.(2π .* x)

u_train = 2.0 .* sin.(4π .* t) .+ 1.0 .* cos.(10π .* t)   # training input
u_test  = 1.5 .* cos.(6π .* t)                            # unseen test input

X      = integrate_fom(x0, u_train, Δt)   # training snapshots (n × K)
X_test = integrate_fom(x0, u_test, Δt)
U      = reshape(u_train, 1, :)           # inputs (m × K)
size(X)

In [ ]:
fig = Figure(size=(900, 330))
ax1 = Axis(fig[1, 1], title="Training trajectory", xlabel="t", ylabel="x")
hm1 = heatmap!(ax1, t, x, X', colormap=:balance)
Colorbar(fig[1, 2], hm1)
ax2 = Axis(fig[1, 3], title="Test trajectory", xlabel="t", ylabel="x")
hm2 = heatmap!(ax2, t, x, X_test', colormap=:balance)
Colorbar(fig[1, 4], hm2)
fig

## Reduced-model utilities and OpInf options

The same `LSOpInfOption` drives batch OpInf and both streaming variants. We use a semi-implicit Euler (`"SI"`) time-derivative approximation, which matches the time stepper of the data, and a small Tikhonov regularization.

In [ ]:
# Semi-implicit Euler integrator for the learned linear-quadratic ROM
function integrate_rom(op, x0hat, u, Δt)
    r = length(x0hat)
    Xhat = zeros(r, length(u))
    Xhat[:, 1] = x0hat
    F = lu(I - Δt * op.A)
    for k in 1:length(u)-1
        xk = Xhat[:, k]
        Xhat[:, k+1] = F \ (xk + Δt * (op.A2u * (xk ⊘ xk) + op.B[:, 1] * u[k+1]))
    end
    return Xhat
end

rel_err(Xref, Xapprox) = norm(Xref - Xapprox) / norm(Xref)

options = LSOpInfOption(
    system=SystemStructure(state=[1, 2], control=1),    # ẋ = A x + A2u (x ⊘ x) + B u
    data=DataStructure(Δt=Δt, deriv_type="SI"),         # semi-implicit Euler derivative
    with_reg=true,
    λ=TikhonovParameter(A=1e-8, A2=1e-6, B=1e-8),
)

## 1. Batch OpInf (reference)

The standard approach needs the whole snapshot matrix in memory to compute the POD basis and the reduced data.

In [ ]:
rmax = 20
r    = 10
F_svd = svd(X)
Vr = F_svd.U[:, 1:r]

op_batch = LnL.opinf(X, Vr, options; U=U)
Xrom = Vr * integrate_rom(op_batch, Vr' * x0, u_train, Δt)
Xrom_test = Vr * integrate_rom(op_batch, Vr' * x0, u_test, Δt)
println("batch OpInf (r=$r): train error = ", rel_err(X, Xrom),
        ", test error = ", rel_err(X_test, Xrom_test))

## 2. Two-pass Streaming-OpInf

**Pass 1** provides the basis $\mathbf{V}_r$. Here we simply reuse the POD basis from above, but any basis computed beforehand works, e.g., from an incremental SVD or from a subset of the data.

**Pass 2** projects each snapshot as it arrives, $\hat{\mathbf{x}}_k = \mathbf{V}_r^\top\mathbf{x}_k$, forms the semi-implicit Euler derivative from the previous reduced snapshot, and calls `stream!`, which performs one recursive least-squares update of the operator matrix. Only $\mathcal{O}(d^2)$ numbers are stored, where $d = r + 1 + r(r+1)/2$ is the number of unknowns per row of the operator matrix.

Because the regularization is set in `options`, the recursive solution after $k$ snapshots *is* the batch Tikhonov-regularized OpInf solution for those $k$ snapshots, with no approximation. We verify this at a few checkpoints for the standard RLS and the inverse QR-decomposition RLS (iQRRLS) algorithms by comparing with batch OpInf on the same data.

In [ ]:
checkpoints = [50; 100:100:K-1; K]

function two_pass(alg; kwargs...)
    solver = TwoPassStreamingOpInf(options=options, n=r, m=1, algorithm=alg; kwargs...)
    op_err = Float64[]
    xhat_prev = Vr' * X[:, 1]
    for k in 2:K
        xhat    = Vr' * X[:, k]                    # reduced snapshot
        xhatdot = (xhat - xhat_prev) / Δt           # semi-implicit Euler derivative
        LnL.stream!(solver, xhat, xhatdot; U=[u_train[k]])
        xhat_prev = xhat
        if k in checkpoints
            # batch OpInf on the first k snapshots, for comparison only
            O_k = LnL.opinf(X[:, 1:k], Vr, options; U=U[:, 1:k]).O
            push!(op_err, norm(Matrix(solver.cache.O') - O_k) / norm(O_k))
        end
    end
    return LnL.terminate_stream(solver), op_err
end

op_rls,    err_rls    = two_pass(:RLS)
op_iqrrls, err_iqrrls = two_pass(:iQRRLS; qr_method=:givens)
for (name, op) in (("RLS", op_rls), ("iQRRLS", op_iqrrls))
    Xs = Vr * integrate_rom(op, Vr' * x0, u_test, Δt)
    println(rpad(name, 7), ": ‖O - O_batch‖/‖O_batch‖ = ", norm(op.O - op_batch.O) / norm(op_batch.O),
            ", test error = ", rel_err(X_test, Xs))
end

In [ ]:
fig = Figure(size=(600, 350))
ax = Axis(fig[1, 1], xlabel="number of streamed snapshots k",
          ylabel="‖O_stream - O_batch(1:k)‖ / ‖O_batch(1:k)‖", yscale=log10,
          title="Two-pass Streaming-OpInf vs. batch OpInf on the same data")
scatterlines!(ax, checkpoints, err_rls, label="RLS")
scatterlines!(ax, checkpoints, err_iqrrls, label="iQRRLS", marker=:xcross, linestyle=:dash)
ylims!(ax, 1e-16, 1e-2)
axislegend(ax)
fig

## 3. One-pass Streaming-OpInf

The one-pass object maintains a truncated incremental SVD $\mathbf{X} \approx \mathbf{V}\boldsymbol{\Sigma}\mathbf{W}^\top$ of rank at most `rmax`. Each call to `stream!` folds one full-order snapshot into the factorization, after which the snapshot can be discarded.

At the end, `stream.V` is the POD basis and the reduced snapshots are $\hat{\mathbf{X}}^\top = \mathbf{W}\boldsymbol{\Sigma}$. The reduced time derivatives are obtained by applying the finite-difference matrix `E` (from `finite_diff_matrix`, consistent with `deriv_type`) to these reduced snapshots, so `compute_stream_operators` can learn a model of *any* dimension `r ≤ rmax` without touching the full-order data again.

In [ ]:
stream = OnePassStreamingOpInf(X[:, 1]; options=options, n_state=n, n_input=1, rank=rmax)
for k in 2:K
    LnL.stream!(stream, X[:, k])     # e.g., read from disk or received from a running solver
end

E, idx = LnL.finite_diff_matrix(options.data.deriv_type, K, Δt)
op_onepass = LnL.compute_stream_operators(stream, E, idx; U=Matrix(U'), rank=r)

Vr_stream = stream.V[:, 1:r]
println("largest principal angle between the bases: ",
        acos(clamp(minimum(svdvals(Vr' * Vr_stream)), -1, 1)))
println("relative singular value error: ", norm(stream.Σ - F_svd.S[1:rmax]) / norm(F_svd.S[1:rmax]))
Xs = Vr_stream * integrate_rom(op_onepass, Vr_stream' * x0, u_test, Δt)
println("one-pass (r=$r): test error = ", rel_err(X_test, Xs))

In [ ]:
fig = Figure(size=(600, 350))
ax = Axis(fig[1, 1], xlabel="index", ylabel="singular value", yscale=log10,
          title="Singular values: batch SVD vs. one-pass iSVD")
scatter!(ax, 1:rmax, F_svd.S[1:rmax], label="batch SVD", markersize=12)
scatter!(ax, 1:rmax, stream.Σ, label="one-pass iSVD", marker=:xcross, markersize=10)
axislegend(ax)
fig

## Accuracy versus reduced dimension

Since the one-pass object keeps rank `rmax`, models of every dimension $r \le$ `rmax` can be extracted after streaming. Below we compare the relative state error on the unseen test input for batch OpInf and one-pass Streaming-OpInf.

In [ ]:
rs = 2:2:rmax
err_batch   = Float64[]
err_onepass = Float64[]
for ri in rs
    V = F_svd.U[:, 1:ri]
    opb = LnL.opinf(X, V, options; U=U)
    push!(err_batch, rel_err(X_test, V * integrate_rom(opb, V' * x0, u_test, Δt)))

    Vs = stream.V[:, 1:ri]
    ops = LnL.compute_stream_operators(stream, E, idx; U=Matrix(U'), rank=ri)
    push!(err_onepass, rel_err(X_test, Vs * integrate_rom(ops, Vs' * x0, u_test, Δt)))
end

fig = Figure(size=(600, 350))
ax = Axis(fig[1, 1], xlabel="reduced dimension r", ylabel="relative state error (test)",
          yscale=log10, title="Test error of the learned ROMs")
scatterlines!(ax, rs, err_batch, label="batch OpInf", markersize=12)
scatterlines!(ax, rs, err_onepass, label="one-pass Streaming-OpInf", marker=:xcross,
              linestyle=:dash, markersize=10)
axislegend(ax, position=:rt)
fig

In [ ]:
Xs = Vr_stream * integrate_rom(op_onepass, Vr_stream' * x0, u_test, Δt)
fig = Figure(size=(900, 330))
ax1 = Axis(fig[1, 1], title="One-pass ROM prediction (test input, r=$r)", xlabel="t", ylabel="x")
hm1 = heatmap!(ax1, t, x, Xs', colormap=:balance)
Colorbar(fig[1, 2], hm1)
ax2 = Axis(fig[1, 3], title="Pointwise error", xlabel="t", ylabel="x")
hm2 = heatmap!(ax2, t, x, abs.(X_test - Xs)', colormap=:viridis)
Colorbar(fig[1, 4], hm2)
fig

## Memory footprint

The point of streaming is that the memory needed for learning does not scale with the full snapshot matrix ($n \times K$). One-pass Streaming-OpInf stores the rank-`rmax` factors ($n \cdot r_{\max} + K \cdot r_{\max}$ numbers), while the two-pass RLS state is independent of both $n$ and $K$. These savings are modest for this small example but become decisive for large-scale simulations with millions of degrees of freedom.

In [ ]:
d = r + 1 + r * (r + 1) ÷ 2          # unknowns per row of the operator matrix
mem_full    = n * K
mem_onepass = n * rmax + K * rmax + rmax
mem_twopass = d^2 + d * r
println("numbers stored — snapshot matrix: $mem_full, one-pass iSVD: $mem_onepass, two-pass RLS: $mem_twopass")

## Summary

- **Two-pass** Streaming-OpInf reproduces the batch Tikhonov-regularized OpInf operators (to about 1e-8 with RLS and 1e-11 with the more robust iQRRLS in this example) while storing only $\mathcal{O}(d^2)$ numbers. Use it when a basis is available and the (reduced) data stream is long.
- **One-pass** Streaming-OpInf builds the POD basis and the reduced data in a single sweep over the full-order data, giving models of any dimension up to `rmax` with essentially the same accuracy as batch OpInf.
- Both reuse `LSOpInfOption` and return `Operators`, so the learned models drop into the same simulation code as batch OpInf.